# Gold Layer — Joined Analytical Fact Table
**What this does:** Joins silver_prices with silver_fear_greed to produce one enriched analytical table. This is the final output that answers the core question: does fear/greed sentiment correlate with price volatility?

**Why Gold exists:** Silver has clean tables but they're separate. A dashboard can't answer "what was the fear score when Bitcoin dropped 5%?" by looking at one table. Gold joins everything together into one flat, query-ready fact table — like the master flight manifest that combines passenger lists, gate assignments, and flight status into one view.

In [0]:
from pyspark.sql import functions as F

# Read both silver tables
df_prices = spark.table("silver_prices")
df_fg     = spark.table("silver_fear_greed")

print(f"Prices rows coming in: {df_prices.count()}")
print(f"Fear & Greed rows coming in: {df_fg.count()}")

In [0]:
# ── CREATE THE JOIN KEY ───────────────────────────────────────────────────────
# prices.last_updated and fear_greed.ingested_at don't share exact timestamps
# because they were fetched a few seconds apart in the same script run.
# Solution: truncate both down to the hour level — both 03:36 and 03:44 become 03:00.
# Then they match on the same "ingestion hour bucket".
#
# date_trunc("hour", timestamp) = rounds a timestamp DOWN to the nearest hour
# Example: 2026-06-17 03:36:30 → 2026-06-17 03:00:00
# Example: 2026-06-17 13:44:37 → 2026-06-17 13:00:00

df_prices_keyed = df_prices.withColumn(
    "ingestion_hour",
    F.date_trunc("hour", F.col("last_updated"))
)

# Only keep the columns we need from fear_greed for the join
df_fg_keyed = df_fg.withColumn(
    "ingestion_hour",
    F.date_trunc("hour", F.col("ingested_at"))
).select(
    "ingestion_hour",
    "fear_greed_value",
    "value_classification"
)

print("Join keys created.")
print("Prices sample:")
display(df_prices_keyed.select("id", "last_updated", "ingestion_hour").limit(5))
print("Fear & Greed sample:")
display(df_fg_keyed.limit(5))

In [0]:
# ── JOIN PRICES WITH FEAR & GREED ─────────────────────────────────────────────
# how="left" = LEFT JOIN — keep ALL price rows even if no fear_greed reading matches
# WHY left and not inner? An inner join would drop price rows that don't have a matching
# fear_greed reading. Left join keeps everything and puts null for missing fear_greed values.
# Safer choice — you never lose data unexpectedly.
df_joined = df_prices_keyed.join(
    df_fg_keyed,
    on="ingestion_hour",
    how="left"
)

print(f"Joined rows: {df_joined.count()}")
display(df_joined.limit(5))

In [0]:
# ── BUILD THE FINAL GOLD FACT TABLE ──────────────────────────────────────────
# Select only the columns that matter for analytics and dashboards
# Add a volatility_category column — bucketing the % change into High/Medium/Low
# WHY categorize? Dashboards and heatmaps work better with labels than raw numbers.
# A chart that shows "High Volatility" vs "Low Volatility" is clearer than "-4.7% vs -0.3%"
#
# F.when().otherwise() = Spark's version of an IF/ELSE statement
# Structure: F.when(condition, value).when(condition2, value2).otherwise(default_value)

df_gold = df_joined.select(

    # Time
    F.col("ingestion_hour"),
    F.col("last_updated"),

    # Coin identity
    F.col("id").alias("coin_id"),
    F.col("symbol"),
    F.col("name"),

    # Price metrics
    F.col("current_price"),
    F.col("market_cap"),
    F.col("market_cap_rank"),
    F.col("total_volume"),
    F.col("high_24h"),
    F.col("low_24h"),
    F.col("price_change_24h"),
    F.col("price_change_percentage_24h"),
    F.col("circulating_supply"),

    # Sentiment from fear & greed
    F.col("fear_greed_value"),
    F.col("value_classification"),

    # Derived column: volatility bucket based on absolute % price change
    # abs() = absolute value — we treat -5% and +5% as equally volatile
    F.when(F.abs(F.col("price_change_percentage_24h")) >= 5, "High")
     .when(F.abs(F.col("price_change_percentage_24h")) >= 2, "Medium")
     .otherwise("Low")
     .alias("volatility_category")
)

print(f"Gold rows: {df_gold.count()}")
display(df_gold)

In [0]:
# ── WRITE TO DELTA LAKE GOLD TABLE ────────────────────────────────────────────
df_gold.write.format("delta").mode("overwrite").saveAsTable("gold_price_sentiment")

print("Gold table written: gold_price_sentiment")